[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/experiments/txt2sql.ipynb)


<!-- SPDX-License-Identifier: CC-BY-NC-SA-4.0 -->

*本 Notebook 的版权归 © [Braintrust Cookbook](https://www.braintrust.dev/docs/cookbook/recipes/Text2SQL-Data) 所有，依据 [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/) 许可协议发布。*

<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
        <br>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">实验：Text2SQL</h1>

构建有效的文本转 SQL 系统，需要严格评估与系统实验。本教程将从零开始，在没有预先准备的问题或预期回复数据集的情况下，完整介绍评估驱动的开发流程。

我们将使用包含近期片名、评分、票房和元数据的电影数据库，演示如何利用 Phoenix 实验框架构建、评估并系统改进文本转 SQL 系统。可以把 Phoenix 看作科学实验室，细致记录每次实验，帮助构建更好的 AI 系统。

In [ ]:
!pip install "arize-phoenix>=11.0.0" openai duckdb datasets pyarrow "pydantic>=2.0.0" openinference-instrumentation-openai --quiet

首先启动 Phoenix 服务，作为评估仪表盘和实验跟踪器。它是观察、衡量和改进文本转 SQL 系统的中心。

注意：如果已经运行 Phoenix 服务，则无需此步骤。

In [ ]:
import phoenix as px

px.launch_app().view()

同时为 OpenAI 配置追踪。追踪是评估驱动开发的关键：Phoenix 能观察文本转 SQL 流程的每一步，捕获输入、输出及延迟和成本等指标，帮助系统改进。

In [ ]:
from phoenix.otel import register

tracer_provider = register(
    endpoint="http://localhost:6006/v1/traces", auto_instrument=True, verbose=False
)  # 为所有 OpenAI 调用插桩

tracer = tracer_provider.get_tracer(__name__)

初始化 Phoenix Client。它是与 Phoenix API 交互的主要入口，可独立于 Phoenix 本体安装。

In [ ]:
from phoenix.client import AsyncClient

px_client = AsyncClient()

最后，确保已设置 OpenAI API 密钥。

In [ ]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("🔑 请输入 OpenAI API 密钥：")

## 下载数据

我们将使用包含近期电影及评分的数据集。选择 DuckDB 作为数据库，以便直接在 Notebook 中运行查询；实际场景中也可以是已有的、包含业务数据的 SQL 数据库。

In [ ]:
import duckdb
from datasets import load_dataset

data = load_dataset("wykonos/movies")["train"]

conn = duckdb.connect(database=":memory:", read_only=False)
conn.register("movies", data.to_pandas())

records = conn.query("SELECT * FROM movies LIMIT 5").to_df().to_dict(orient="records")

for record in records:
    print(record)

## 实现 Text2SQL

先实现简单的文本转 SQL 逻辑。

In [ ]:
import openai

openai_client = openai.AsyncClient()

columns = conn.query("DESCRIBE movies").to_df().to_dict(orient="records")

# 先使用 GPT4o
TASK_MODEL = "gpt-4o"
CONFIG = {"model": TASK_MODEL}


system_prompt = (
    "你是 SQL 专家，给定一个名为 movies 的表，包含以下列：\n"
    f"{','.join(column['column_name'] + ': ' + column['column_type'] for column in columns)}\n"
    "编写与用户请求对应的 SQL 查询。仅返回查询文本，"
    "不要反引号、Markdown 等格式。"
)


@tracer.chain
async def generate_query(input):
    response = await openai_client.chat.completions.create(
        model=TASK_MODEL,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": input,
            },
        ],
    )
    return response.choices[0].message.content

In [ ]:
query = await generate_query("what was the most popular movie?")
print(query)

看起来已经可以生成 SQL！尝试运行查询，看看是否得到预期结果。

In [ ]:
@tracer.tool
def execute_query(query):
    return conn.query(query).fetchdf().to_dict(orient="records")


execute_query(query)

## 评估的三大支柱

有效的 AI 评估依赖三个基础：

1. **数据**：代表真实使用场景的精选示例
2. **任务**：被评估的实际函数或工作流
3. **评估器**：性能的量化度量

首先创建**数据**，即一组希望文本转 SQL 系统能正确处理的电影相关问题。

In [ ]:
questions = [
    "Which Brad Pitt movie received the highest rating?",
    "What is the top grossing Marvel movie?",
    "What foreign-language fantasy movie was the most popular?",
    "what are the best sci-fi movies of 2017?",
    "What anime topped the box office in the 2010s?",
    "Recommend a romcom that stars Paul Rudd.",
]

将上面的数据作为版本化数据集存储到 Phoenix。

In [ ]:
ds = await px_client.datasets.create_dataset(
    name="movie-example-questions",
    inputs=[{"question": question} for question in questions],
)

接下来定义任务：根据自然语言问题生成 SQL 查询。

In [ ]:
@tracer.chain
async def text2sql(question):
    query = await generate_query(question)
    results = None
    error = None
    try:
        results = execute_query(query)
    except duckdb.Error as e:
        error = str(e)

    return {
        "query": query,
        "results": results,
        "error": error,
    }

最后定义评估分数。使用以下简单函数判断生成的 SQL 查询是否正确。这里 `has_results` 是一个合适指标，因为我们知道数据集中的所有问题都能通过 SQL 回答。

In [ ]:
# 测试 SQL 执行是否无错误


def no_error(output):
    return 1.0 if output.get("error") is None else 0.0


# 测试查询是否有结果
def has_results(output):
    results = output.get("results")
    has_results = results is not None and len(results) > 0
    return 1.0 if has_results else 0.0

现在运行评估实验。

In [ ]:
from typing import Any


async def task(input: Any):
    return await text2sql(input["question"])


experiment = await px_client.experiments.run_experiment(
    dataset=ds, task=task, evaluators=[no_error, has_results], experiment_metadata=CONFIG
)

In [ ]:
def show_evaluation_summary(exp):
    no_error_scores = [
        run.result["score"]
        for run in exp["evaluation_runs"]
        if run.result and "score" in run.result and run.name == "no_error"
    ]
    has_results_scores = [
        run.result["score"]
        for run in exp["evaluation_runs"]
        if run.result and "score" in run.result and run.name == "has_results"
    ]

    print("📊 评估结果：")
    if no_error_scores:
        avg_no_error = sum(no_error_scores) / len(no_error_scores)
        print(f"  无错误： {avg_no_error:.3f} (n={len(no_error_scores)})")

    if has_results_scores:
        avg_has_results = sum(has_results_scores) / len(has_results_scores)
        print(f"  有结果： {avg_has_results:.3f} (n={len(has_results_scores)})")


show_evaluation_summary(experiment)

查看基线模型对电影问题的表现。分析成功查询与失败情况，理解需要改进的地方。

## 解读结果

初始评估已完成，下面分析失败可能由什么引起。

查看没有结果的查询，类型相关查询可能因为模型不知道类型的存储形式而失败，例如 "Sci-Fi" 与 "Science Fiction" 的差别。

向模型展示数据样本，即少样本示例，可能改善这类问题，因为它们会告诉 LLM 哪些内容可以查询。

尝试使用少样本示例改进提示词，看看能否得到更好结果。

In [ ]:
samples = conn.query("SELECT * FROM movies LIMIT 5").to_df().to_dict(orient="records")

example_row = "\n".join(
    f"{column['column_name']} | {column['column_type']} | {samples[0][column['column_name']]}"
    for column in columns
)

column_header = " | ".join(column["column_name"] for column in columns)

few_shot_examples = "\n".join(
    " | ".join(str(sample[column["column_name"]]) for column in columns) for sample in samples
)

system_prompt = (
    "你是 SQL 专家，给定一个名为 `movies` 的表，包含以下列：\n\n"
    "Column | Type | Example\n"
    "-------|------|--------\n"
    f"{example_row}\n"
    "\n"
    "Examples:\n"
    f"{column_header}\n"
    f"{few_shot_examples}\n"
    "\n"
    "编写与用户请求对应的 DuckDB SQL 查询。"
    "Return just the query text, 不要反引号、Markdown 等格式。"
)


async def generate_query(input):
    response = await openai_client.chat.completions.create(
        model=TASK_MODEL,
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": input,
            },
        ],
    )
    return response.choices[0].message.content


print(await generate_query("what are the best sci-fi movies in the 2000s?"))

效果好多了！最后添加评分函数，在结果存在时将其与预期结果比较。


In [ ]:
experiment = await px_client.experiments.run_experiment(
    dataset=ds, task=task, evaluators=[has_results, no_error], experiment_metadata=CONFIG
)

In [ ]:
show_evaluation_summary(experiment)

LLM 看起来已能为所有问题生成有效查询。试试使用 LLM 裁判，看看它评判结果的能力如何。

In [ ]:
import json

from phoenix.client.experiments import create_evaluator

judge_instructions = """
你是裁判，判断给定问题能否通过提供的 SQL 查询与结果回答。
务必确认 SQL 查询准确对应问题。

如果 SQL 查询和结果准确回答问题，给出标签 `correct`。
如果 SQL 查询不对应问题或无效，给出标签 `invalid`。
"""


@create_evaluator(kind="llm")
async def qa_correctness(input: Any, output: Any) -> float:
    question = input.get("question")
    query = output.get("query")
    results = output.get("results")
    response = await openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": judge_instructions},
            {
                "role": "user",
                "content": f"Question: {question}\nSQL Query: {query}\nSQL Results: {results}",
            },
        ],
        tool_choice="required",
        tools=[
            {
                "type": "function",
                "function": {
                    "name": "qa_correctness",
                    "description": "判断 SQL 查询和结果是否准确回答问题。",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "explanation": {
                                "type": "string",
                                "description": "解释标签为何是 correct 或 invalid。",
                            },
                            "label": {"type": "string", "enum": ["correct", "invalid"]},
                        },
                    },
                },
            }
        ],
    )
    if response.choices[0].message.tool_calls is None:
        raise ValueError("回复中未找到工具调用")
    args = json.loads(response.choices[0].message.tool_calls[0].function.arguments)
    label = args["label"]
    score = 1 if label == "correct" else 0
    return score


updated_experiment = await px_client.experiments.evaluate_experiment(
    experiment=experiment, evaluators=[qa_correctness]
)


def show_qa_correctness_results(exp):
    qa_correctness_scores = [
        run.result["score"]
        for run in exp["evaluation_runs"]
        if run.result and "score" in run.result and run.name == "qa_correctness"
    ]

    print("📊 QA Correctness 评估结果：")
    if qa_correctness_scores:
        avg_correctness = sum(qa_correctness_scores) / len(qa_correctness_scores)
        print(f"  问答正确性： {avg_correctness:.3f} (n={len(qa_correctness_scores)})")


show_qa_correctness_results(updated_experiment)

LLM 裁判评分与人工评估高度一致，说明它可以作为有效的自动评估方法。当传统规则评分函数难以实现时，这种方法尤其有价值。

LLM 裁判也展现了理解细节的优势，例如正确区分 'Anime' 和 'Animation' 是不同类型，而代码评估器遗漏了这一细节。这说明，为特定任务需求定制 LLM 裁判，对准确评估非常重要。

我们已经有了根据自然语言问题生成 SQL 查询的简单流程。Phoenix 一直追踪整个流程，因此可以通过 Phoenix UI，将生成成功查询的 span 转换成**黄金数据集**中的示例，用于回归测试。

## 生成更多数据

让模型描述数据集中的已有 SQL 查询，生成一些训练数据。


In [ ]:
from typing import List

from pydantic import BaseModel


class Question(BaseModel):
    sql: str
    question: str


class Questions(BaseModel):
    questions: List[Question]


sample_rows = "\n".join(
    f"{column['column_name']} | {column['column_type']} | {samples[0][column['column_name']]}"
    for column in columns
)
synthetic_data_prompt = f"""你是 SQL 专家，给定一个名为 movies 的表，包含以下列：

列 | 类型 | 示例
-------|------|--------
{sample_rows}

生成一些值得针对该表提问的 SQL 查询。返回 SQL 查询字符串，
以及该查询回答的问题。问题应有清晰边界，既不要太宽泛，也不要太狭窄。"""

response = await openai_client.chat.completions.create(
    model="gpt-4o",
    temperature=0,
    messages=[
        {
            "role": "user",
            "content": synthetic_data_prompt,
        }
    ],
    tools=[
        {
            "type": "function",
            "function": {
                "name": "generate_questions",
                "description": "生成一些值得针对该表提问的 SQL 查询。",
                "parameters": Questions.model_json_schema(),
            },
        }
    ],
    tool_choice={"type": "function", "function": {"name": "generate_questions"}},
)

assert response.choices[0].message.tool_calls is not None
generated_questions = json.loads(response.choices[0].message.tool_calls[0].function.arguments)[
    "questions"
]

print("生成的问题数量：", len(generated_questions))
print("First question: ", generated_questions[0])

In [ ]:
generated_dataset = []
for q in generated_questions:
    try:
        result = execute_query(q["sql"])
        example = {
            "input": q["question"],
            "expected": {
                "results": result or [],
                "query": q["sql"],
            },
            "metadata": {
                "category": "Generated",
            },
        }
        print(example)
        generated_dataset.append(example)
    except duckdb.Error as e:
        print(f"Query failed: {q['sql']}", e)
        print("Skipping...")

generated_dataset[0]

使用新的合成数据创建数据集。


In [ ]:
synthetic_dataset = await px_client.datasets.create_dataset(
    name="movies-golden-synthetic",
    inputs=[{"question": example["input"]} for example in generated_dataset],
    outputs=[example["expected"] for example in generated_dataset],
)

In [ ]:
exp = await px_client.experiments.run_experiment(
    dataset=synthetic_dataset,
    task=task,
    evaluators=[no_error, has_results],
    experiment_metadata=CONFIG,
    timeout=60,
)

In [ ]:
show_evaluation_summary(exp)

现在有了更多数据。可通过以下方式继续改进：

- 检查生成数据中的问题
- 完善提示词
- 将错误展示给模型

这构成持续改进系统的流程。

## 结语

本教程构建了用于查询电影数据的文本转 SQL 系统。从基本示例和评估器开始，通过增加少样本示例、使用 LLM 裁判评估来改进表现。

要点：

- 从简单评估器开始，发现基础问题
- 使用少样本示例提高准确性
- 使用 LLM 生成更多训练数据
- 通过 Phoenix 实验跟踪进展

还可以添加更好的评估器或处理边界情况，继续改进系统。